In [ ]:
# Standard Library
import os
import random
from pathlib import Path

# Numerical Computing & Data Handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Image Processing
from PIL import Image

# TorchVision
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
# Data Loading Utilities
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import classification_report, confusion_matrix

from tqdm import tqdm

## Configuration

In [2]:
DATASET_PATH = Path("data/raw/PlantVillage")

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
DATASET_PATH.iterdir()

<generator object Path.iterdir at 0x00000293FF5BAC70>

## Image Preprocessing

In [7]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [8]:
full_dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=train_transforms
)

class_names = full_dataset.classes

print(f"Total Classes : {len(class_names)}")
print(class_names)

Total Classes : 15
['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [9]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(val_dataset)}")

Training Images   : 16510
Validation Images : 4128
